# Revenue & Risk Driver Analysis – NorthRiver Analytics

This notebook analyzes which customers drive revenue and where payment risk is concentrated for the NorthRiver Analytics B2B SaaS simulation.

## Goals
- Summarize top-customer revenue concentration using `../data/top10_customer_concentration.csv`
- Summarize overdue/payment risk for those same customers using `../data/top_customer_overdue_risk.csv`
- Connect these drivers back to overall revenue trends in `../data/monthly_financial_kpis.csv`
- Produce clear, business-focused findings suitable for an Operations/Data Analyst role
- Export clean summary tables for visualization in Power BI or Excel.


## 1. Imports and data loading

In this project, all raw and intermediate CSV files live in the `data/` folder one level above this notebook. We load three key datasets:

- `monthly_financial_kpis.csv`: monthly revenue, cost, and margin trends for the whole business.
- `top10_customer_concentration.csv`: top 10 customers by revenue with each customer's revenue share.
- `top_customer_overdue_risk.csv`: the same top customers with summary overdue/late payment metrics.

This setup mirrors a typical operations analytics workflow where an analyst pulls from curated tables or exports maintained by upstream systems.


In [2]:
import pandas as pd

# Load monthly KPIs (no header row in the CSV, so we assign names explicitly)
monthly_cols = ["year", "month", "month_name", "revenue", "cost", "margin", "margin_pct"]
monthly_kpis = pd.read_csv("../data/monthly_financial_kpis.csv", names=monthly_cols)

# Load top-10 customer revenue concentration.
# Columns (based on the SQL that generated the file):
#   customer_id, customer_name, segment, industry, region, total_revenue, revenue_share
concentration_cols = [
    "customer_id",
    "customer_name",
    "segment",
    "industry",
    "region",
    "total_revenue",
    "revenue_share"
]
customer_conc = pd.read_csv("../data/top10_customer_concentration.csv", names=concentration_cols)

# Load overdue / payment risk metrics for the same customers.
# Columns:
#   customer_id, customer_name, segment, industry, region,
#   total_revenue, revenue_share,
#   total_invoices, overdue_invoices, overdue_amount,
#   overdue_ratio_by_invoice, overdue_ratio_by_amount
risk_cols = [
    "customer_id",
    "customer_name",
    "segment",
    "industry",
    "region",
    "total_revenue",
    "revenue_share",
    "total_invoices",
    "overdue_invoices",
    "overdue_amount",
    "overdue_ratio_by_invoice",
    "overdue_ratio_by_amount"
]
customer_risk = pd.read_csv("../data/top_customer_overdue_risk.csv", names=risk_cols)

monthly_kpis.head(), customer_conc.head(), customer_risk.head()


(   year  month month_name  revenue     cost   margin  margin_pct
 0  2023      3      March   1940.0  1069.16   870.84    0.448886
 1  2023      4      April   5340.0  2965.60  2374.40    0.444644
 2  2023      5        May   5518.0  2987.40  2530.60    0.458608
 3  2023      6       June   5814.0  3229.66  2584.34    0.444502
 4  2023      7       July   7967.0  4502.94  3464.06    0.434801,
    customer_id customer_name segment       industry         region  \
 0           19   Customer_19   Small           Tech  North America   
 1            5    Customer_5   Small  Manufacturing         Europe   
 2           15   Customer_15   Small  Manufacturing           APAC   
 3           12   Customer_12   Small        Finance           APAC   
 4           22   Customer_22   Small        Finance  North America   
 
    total_revenue  revenue_share  
 0       175319.0       0.188370  
 1       116071.0       0.124711  
 2        85241.0       0.091586  
 3        78368.0       0.084202  


## 2. Revenue trend context

Before looking at individual customers, it is important to understand how total revenue has behaved over time. This mirrors how an operations or data analyst would first look at a KPI dashboard before drilling into drivers.

We:
- Build a proper `period` column from `year` and `month`.
- Sort the data chronologically.
- Calculate a simple year-over-year comparison where possible.


In [3]:
# Build a period column and sort chronologically
monthly_kpis["period"] = pd.to_datetime(
    monthly_kpis["year"].astype(str) + "-" + monthly_kpis["month"].astype(str),
    format="%Y-%m"
)
monthly_kpis = monthly_kpis.sort_values("period").reset_index(drop=True)

# Year-over-year comparison for months where prior-year data exists
monthly_kpis["revenue_yoy"] = monthly_kpis["revenue"].pct_change(periods=12)

monthly_kpis[["period", "revenue", "revenue_yoy"]].tail(12)


,period,revenue,revenue_yoy
25,2025-04-01,30558.0,0.177981
26,2025-05-01,30317.0,0.059405
27,2025-06-01,35004.0,0.112085
28,2025-07-01,34806.0,0.181868
29,2025-08-01,36556.0,0.193159
30,2025-09-01,39328.0,0.352035
31,2025-10-01,43906.0,0.490866
32,2025-11-01,41250.0,0.465624
33,2025-12-01,44761.0,0.490245
34,2026-01-01,42625.0,0.527778


### Interpretation – revenue trend

- Total monthly revenue is growing strongly year over year for most of the last 12 months. From mid‑2024 through early 2026, year‑over‑year changes are often in the high teens to 50% range, which is consistent with a healthy, scaling B2B SaaS business.<br>

- March 2026 is a clear outlier: revenue drops from about 29k in March 2025 to 16.5k in March 2026 (roughly a −43% YoY change). That kind of one‑month swing is unlikely to be random noise and looks more like an abnormal event (for example, a large customer churn, a pricing change, or a billing issue).<br>

- When explaining this in a business context, it is important to separate the overall growth story (“the business is trending up”) from the one‑off shock (“there was a specific event in March 2026 that breaks the pattern”), and to recommend follow‑up investigation into that month.

## 3. Top-10 customer revenue concentration

Next, we focus on the top-10 customers by revenue. The goal is to answer:

- How much of total top-customer revenue is concentrated in these accounts?
- Is there a single customer that represents a disproportionate share?
- How diversified are segment/industry/region among the top accounts?

These are classic concentration questions that matter for both financial risk and operational planning.


In [4]:
# Sanity checks on concentration data
customer_conc_sorted = customer_conc.sort_values("total_revenue", ascending=False).reset_index(drop=True)

# Total revenue and total share across the top-10 customers
total_top10_revenue = customer_conc_sorted["total_revenue"].sum()
total_top10_share = customer_conc_sorted["revenue_share"].sum()

largest_customer = customer_conc_sorted.iloc[0]

summary_concentration = {
    "top10_total_revenue": total_top10_revenue,
    "top10_total_share": total_top10_share,
    "largest_customer_name": largest_customer["customer_name"],
    "largest_customer_revenue": largest_customer["total_revenue"],
    "largest_customer_share": largest_customer["revenue_share"]
}

summary_concentration, customer_conc_sorted


({'top10_total_revenue': np.float64(775090.0),
  'top10_total_share': np.float64(0.832786),
  'largest_customer_name': 'Customer_19',
  'largest_customer_revenue': np.float64(175319.0),
  'largest_customer_share': np.float64(0.18837)},
    customer_id customer_name segment       industry         region  \
 0           19   Customer_19   Small           Tech  North America   
 1            5    Customer_5   Small  Manufacturing         Europe   
 2           15   Customer_15   Small  Manufacturing           APAC   
 3           12   Customer_12   Small        Finance           APAC   
 4           22   Customer_22   Small        Finance  North America   
 5            4    Customer_4   Small           Tech  North America   
 6           29   Customer_29   Small           Tech         Europe   
 7            3    Customer_3   Small         Retail           APAC   
 8           24   Customer_24   Small           Tech           APAC   
 9           27   Customer_27   Small        Finance  

### Interpretation – top-10 customer revenue concentration

- Revenue is meaningfully concentrated in a small set of accounts. Within the top‑10 cohort, the top 5 customers (Customers 19, 5, 15, 12, and 22) together account for just over 58% of the total revenue captured in this table.<br>
- Customer_19 is the single largest account, with total revenue of 175,319 and a revenue_share of 0.18837, so it contributes about 18.8% of top‑10 revenue on its own. Customer_5 adds another 12.5%, and Customer_15 contributes about 9.2%, which shows how heavily the business leans on a few key customers.<br>

- All top‑10 customers are in the Small segment but are diversified across industries (Tech, Manufacturing, Finance, Retail) and regions (North America, Europe, APAC), so exposure is not limited to a single vertical or geography. At the same time, any disruption or churn among a small cluster of Small Tech and Finance customers would have an outsized impact on performance.

## 4. Overdue / payment risk among top customers

Now we bring in overdue and late-payment behavior for the same customers. The key questions:

- Among the top-revenue customers, who has the highest share of overdue invoices?
- Who has the largest **overdue amount** relative to their total revenue?
- Are there customers that are both high-revenue and high-risk?

This is the kind of analysis that supports LEAN / DMAIC work around collections and cash-flow reliability.


In [5]:
# Combine revenue concentration and overdue risk on customer_id
combined = customer_risk.copy().sort_values("total_revenue", ascending=False).reset_index(drop=True)

# Create helper metrics for easier interpretation
combined["overdue_pct_of_revenue"] = combined["overdue_amount"] / combined["total_revenue"]
combined["overdue_invoice_rate"] = combined["overdue_invoices"] / combined["total_invoices"]

# Sort to see which customers are most concerning on different dimensions
by_overdue_amount = combined.sort_values("overdue_amount", ascending=False)
by_overdue_pct_revenue = combined.sort_values("overdue_pct_of_revenue", ascending=False)
by_overdue_invoice_rate = combined.sort_values("overdue_invoice_rate", ascending=False)

combined.head(), by_overdue_amount.head(), by_overdue_pct_revenue.head(), by_overdue_invoice_rate.head()


(   customer_id customer_name segment       industry         region  \
 0           19   Customer_19   Small           Tech  North America   
 1            5    Customer_5   Small  Manufacturing         Europe   
 2           15   Customer_15   Small  Manufacturing           APAC   
 3           12   Customer_12   Small        Finance           APAC   
 4           22   Customer_22   Small        Finance  North America   
 
    total_revenue  revenue_share  total_invoices  overdue_invoices  \
 0       175319.0       0.188370             881                86   
 1       116071.0       0.124711             779                95   
 2        85241.0       0.091586            1079               100   
 3        78368.0       0.084202             992                86   
 4        75735.0       0.081373             765                83   
 
    overdue_amount  overdue_ratio_by_invoice  overdue_ratio_by_amount  \
 0         17114.0                  0.097616                 0.097616   
 1  

### Interpretation – overdue / payment risk

- Every top‑10 customer shows some level of overdue behavior, with overdue invoice ratios by count and amount typically in the 8–12% range. That means even the best customers are generating a non‑trivial volume of late payments, which has direct implications for cash‑flow planning.<br>

- Customer_19, the largest revenue contributor, has an overdue_amount of 17,114 and an overdue_ratio_by_amount of about 9.8%, so nearly one‑tenth of its billed revenue is overdue at a point in time. This makes it both a key revenue driver and the single largest contributor to overdue balances.<br>

- Customer_5 stands out as the riskiest payer in relative terms, with overdue_ratio_by_invoice and overdue_ratio_by_amount of roughly 12.2%, and an overdue_amount of 14,155. In practical terms, about one in eight of its invoices are late and around 12% of its revenue is overdue, which is high for a top account.<br>

- Customer_29, while smaller in total revenue (60,984), also has overdue rates around 11.5%, showing that mid‑tier customers can be structurally late payers as well and shouldn’t be ignored in collections strategy.

## 5. Export summary tables for reporting

To make this work reusable in tools like Power BI or Excel—which is exactly what an Operations Data Analyst would be expected to do—we export two clean summary tables:

- `customer_revenue_concentration_summary.csv` – top-10 customers with revenue and share.
- `customer_overdue_risk_summary.csv` – the same customers with overdue metrics.

These can be joined or visualized to show where revenue is concentrated and which accounts carry the most payment risk.


In [6]:
# Keep a tidy version of the top-10 concentration table
conc_export = customer_conc_sorted.copy()
conc_export.to_csv("../data/customer_revenue_concentration_summary.csv", index=False)

# Keep a tidy version of the overdue risk metrics with helper percentages
risk_export = combined[[
    "customer_id",
    "customer_name",
    "segment",
    "industry",
    "region",
    "total_revenue",
    "revenue_share",
    "total_invoices",
    "overdue_invoices",
    "overdue_amount",
    "overdue_ratio_by_invoice",
    "overdue_ratio_by_amount",
    "overdue_pct_of_revenue",
    "overdue_invoice_rate"
]].copy()

risk_export.to_csv("../data/customer_overdue_risk_summary.csv", index=False)

"Saved ../data/customer_revenue_concentration_summary.csv and ../data/customer_overdue_risk_summary.csv"


'Saved ../data/customer_revenue_concentration_summary.csv and ../data/customer_overdue_risk_summary.csv'

## 6. How to explain this analysis

- I started at the portfolio level by looking at monthly revenue and year‑over‑year growth, then drilled into the top 10 customers to understand which accounts actually drive the business.<br>

- I quantified revenue concentration by customer, segment, industry, and region, showing that a small group of Small‑segment customers (especially in Tech and Finance across North America and APAC) account for most of the top‑customer revenue.<br>  

- I then joined customer‑level revenue with invoice‑level overdue behavior to identify accounts that are both high‑revenue and high‑risk, using metrics like overdue percentage of revenue and overdue invoice rate to prioritize where collections and process improvements would have the most impact.<br>  

- Finally, I exported clean summary tables that can be dropped into Power BI or Excel, so stakeholders can monitor concentration and payment risk on an ongoing basis rather than as a one‑off analysis.